In [ ]:
# ============================================================
# Disease severity analysis
# Two-sided Mann–Whitney U test with permutation-based P values
#
# Comparison: Mock vs V1/C3dsR
# Tested separately at D12, D21, and D27
#
# Appropriate here because:
#   - n = 3 plants per treatment
#   - disease score is ordinal (0–4)
#   - many tied values are present
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu, PermutationMethod


# ============================================================
# 1. INPUT DATA
# ============================================================

scores = {
    "D12": {
        "mock": [2, 2, 2],
        "V1/C3dsR": [1, 2, 2]
    },
    "D21": {
        "mock": [3, 3, 3],
        "V1/C3dsR": [3, 3, 3]
    },
    "D27": {
        "mock": [4, 4, 4],
        "V1/C3dsR": [3, 4, 4]
    }
}

day_order = ["D12", "D21", "D27"]

day_numeric = {
    "D12": 12,
    "D21": 21,
    "D27": 27
}


# ============================================================
# 2. CONVERT TO DATAFRAME
# ============================================================

rows = []

for day in day_order:

    for treatment, vals in scores[day].items():

        for rep, score in enumerate(vals, start=1):

            rows.append({
                "day": day,
                "day_num": day_numeric[day],
                "treatment": treatment,
                "rep": rep,
                "score": score
            })


df = pd.DataFrame(rows)


print("\nRaw data:")
print(df.to_string(index=False))


# ============================================================
# 3. SUMMARY STATISTICS
# ============================================================

summary = (
    df.groupby(["day", "treatment"], sort=False)
    .agg(
        n=("score", "count"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        sd=("score", "std"),
        minimum=("score", "min"),
        maximum=("score", "max")
    )
    .reset_index()
)


print("\nSummary statistics:")
print(summary.to_string(index=False))


# ============================================================
# 4. DEFINE PERMUTATION METHOD
# ============================================================

# With n = 3 vs 3, there are only:
#
#       6 choose 3 = 20
#
# possible assignments of six observations into two groups
# of three.
#
# n_resamples = 10000 is therefore more than sufficient for
# SciPy to evaluate the complete permutation distribution.
#
# random_state is included for reproducibility.

permutation_method = PermutationMethod(
    n_resamples=10000,
    random_state=42
)


# ============================================================
# 5. MANN–WHITNEY U TEST WITH PERMUTATION P VALUES
# ============================================================

stat_rows = []


for day in day_order:

    mock_vals = (
        df[
            (df["day"] == day) &
            (df["treatment"] == "mock")
        ]["score"]
        .to_numpy()
    )

    dsr_vals = (
        df[
            (df["day"] == day) &
            (df["treatment"] == "V1/C3dsR")
        ]["score"]
        .to_numpy()
    )


    # --------------------------------------------------------
    # Mann–Whitney U statistic
    # P value calculated by permutation
    # --------------------------------------------------------

    result = mannwhitneyu(
        mock_vals,
        dsr_vals,
        alternative="two-sided",
        method=permutation_method
    )


    stat_rows.append({

        "day": day,

        "mock_values":
            ",".join(map(str, mock_vals)),

        "V1/C3dsR_values":
            ",".join(map(str, dsr_vals)),

        "U_statistic":
            result.statistic,

        "p_value":
            result.pvalue
    })


stats_df = pd.DataFrame(stat_rows)


print("\nPermutation Mann–Whitney U tests:")
print(stats_df.to_string(index=False))


# ============================================================
# 6. P-VALUE LABEL FUNCTION
# ============================================================

def p_label(p):

    if p < 0.001:
        return "p < 0.001"

    elif p < 0.01:
        return f"p = {p:.3f}"

    else:
        return f"p = {p:.2f}"


# ============================================================
# 7. PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 4.8)
)


# ------------------------------------------------------------
# X-axis positions
# ------------------------------------------------------------

x_positions = {
    "D12": 0,
    "D21": 1,
    "D27": 2
}


# Slight horizontal separation between treatments

offsets = {
    "mock": -0.08,
    "V1/C3dsR": 0.08
}


# Colours

colors = {
    "mock": "#d62728",
    "V1/C3dsR": "#7f7f7f"
}


# ============================================================
# 8. PLOT INDIVIDUAL DATA POINTS
# ============================================================

for treatment in ["mock", "V1/C3dsR"]:

    sub_t = df[
        df["treatment"] == treatment
    ]


    for day in day_order:

        vals = (
            sub_t[
                sub_t["day"] == day
            ]["score"]
            .to_numpy()
        )


        x0 = (
            x_positions[day]
            + offsets[treatment]
        )


        # Deterministic horizontal jitter
        # This separates identical points visually.

        jitter = np.linspace(
            -0.02,
            0.02,
            len(vals)
        )


        ax.scatter(

            np.full(
                len(vals),
                x0
            ) + jitter,

            vals,

            s=55,

            color=colors[treatment],

            alpha=0.70,

            zorder=3
        )


# ============================================================
# 9. PLOT MEAN DISEASE SCORE
# ============================================================

for treatment in ["mock", "V1/C3dsR"]:

    sub_t = df[
        df["treatment"] == treatment
    ]


    means = (
        sub_t
        .groupby("day")["score"]
        .mean()
        .reindex(day_order)
    )


    xs = [
        x_positions[d]
        + offsets[treatment]

        for d in day_order
    ]


    ax.plot(

        xs,

        means,

        marker="o",

        markersize=8,

        linewidth=2.8,

        color=colors[treatment],

        label=treatment,

        zorder=4
    )


# ============================================================
# 10. ADD P VALUES
# ============================================================

for day in day_order:

    day_df = df[
        df["day"] == day
    ]


    y_max = day_df["score"].max()


    p = stats_df.loc[
        stats_df["day"] == day,
        "p_value"
    ].iloc[0]


    x_left = (
        x_positions[day]
        + offsets["mock"]
    )


    x_right = (
        x_positions[day]
        + offsets["V1/C3dsR"]
    )


    x_center = (
        x_left + x_right
    ) / 2


    # Height of significance bracket

    y = y_max + 0.22

    h = 0.08


    # Draw bracket

    ax.plot(

        [
            x_left,
            x_left,
            x_right,
            x_right
        ],

        [
            y,
            y + h,
            y + h,
            y
        ],

        color="black",

        linewidth=1
    )


    # Add P-value text

    ax.text(

        x_center,

        y + h + 0.03,

        p_label(p),

        ha="center",

        va="bottom",

        fontsize=10
    )


# ============================================================
# 11. AXIS FORMATTING
# ============================================================

ax.set_xticks(
    [0, 1, 2]
)


ax.set_xticklabels(
    day_order,
    fontsize=12
)


ax.set_ylabel(
    "Disease score",
    fontsize=14
)


ax.set_xlabel(
    "Time point",
    fontsize=14
)


ax.set_ylim(
    0,
    4.6
)


ax.set_title(
    "Disease score over time",
    fontsize=18,
    pad=12
)


ax.legend(
    frameon=False,
    fontsize=12
)


ax.spines["top"].set_visible(False)

ax.spines["right"].set_visible(False)


plt.tight_layout()


# ============================================================
# 12. SAVE FIGURE
# ============================================================

plt.savefig(
    "Disease_score_permutation_MWU.png",
    dpi=600,
    bbox_inches="tight"
)


plt.savefig(
    "Disease_score_permutation_MWU.pdf",
    dpi=600,
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 13. SAVE STATISTICAL RESULTS
# ============================================================

stats_df.to_csv(
    "Disease_score_permutation_MWU_statistics.csv",
    index=False
)


summary.to_csv(
    "Disease_score_summary.csv",
    index=False
)


print("\nAnalysis complete.")